# Leçon 16 - Déployer des agents évolutifs avec Microsoft Foundry

Dans ce cahier, vous construisez un **agent de support client prêt pour la production** pour l’entreprise fictive **Contoso**. Contrairement aux leçons précédentes, l’objectif ici n’est pas la boucle de raisonnement de l’agent — c’est tout ce qui l’entoure *autour* qui rend un agent sûr à déployer à grande échelle :

1. **Appel d’outils** — consultations de commandes et création de tickets.
2. **RAG** — réponses politiques à partir d’une base de connaissances.
3. **Mémoire** — se souvenir du client entre les échanges.
4. **Routage de modèle** — envoyer les demandes simples à un petit modèle, les complexes à un grand modèle.
5. **Mise en cache des réponses** — servir les questions répétées sans appel au modèle.
6. **Approbation humaine** — les remboursements au-delà d’un seuil sont mis en pause pour validation.
7. **Portail d’évaluation** — un jeu de test hors ligne qui bloque une mauvaise version.
8. **Observabilité** — traçage OpenTelemetry autour de chaque requête.

Chaque section est autonome et exécutable. Lisez chaque ligne — les primitives de production sont volontairement réduites.


## Configuration

Avant d'exécuter ce notebook, assurez-vous d'avoir :

1. **Un projet Microsoft Foundry** avec un modèle de chat déployé (par ex. `gpt-5-mini`).
2. **Connecté avec l'interface Azure CLI** — exécutez `az login` dans votre terminal.
3. **Défini les variables d’environnement requises :**
   - `AZURE_AI_PROJECT_ENDPOINT` — l’endpoint de votre projet Microsoft Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — le nom de votre modèle déployé.

La section RAG utilise **Azure AI Search** lorsque `AZURE_SEARCH_SERVICE_ENDPOINT` et `AZURE_SEARCH_API_KEY` sont définis, et revient à une recherche en mémoire pour que le notebook fonctionne sans ressource Search.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. Outils

Les outils de production effectuent un travail réel sur des systèmes réels. Ici, nous simulons une base de données de commandes et un système de billetterie avec des fonctions Python simples. Le décorateur `@tool` les expose à l'agent.

Notez que `issue_refund` utilise `approval_mode="always_require"` pour les remboursements au-delà d'un seuil — c'est la primitive humain-dans-la-boucle que nous déployons plus tard.


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — Base de Connaissances des Politiques

Les questions sur les politiques ("quel est votre délai de retour ?") doivent être répondues à partir d'une source faisant autorité, et non de la mémoire du modèle. Nous encapsulons une petite base de connaissances comme un outil de recherche.

En production, il s'agit de **Azure AI Search** ; ici, nous fournissons une recherche par mots-clés en mémoire afin que le notebook fonctionne partout, et passons automatiquement à Azure AI Search lorsque les variables d'environnement sont présentes.


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. Mémoire

Un agent de support qui oublie avec qui il parle est un mauvais agent de support. Nous gardons un petit magasin de profils par client et injectons un court résumé dans les instructions de l'agent. En production, c'est un service de mémoire (voir la Leçon 13) ; ici, un dictionnaire rend le modèle visible.


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 & 5. Routage de modèle et mise en cache des réponses

Deux leviers de coût connectés à un seul gestionnaire de requêtes :

- **Routage** : un classificateur heuristique peu coûteux détermine si une requête nécessite le petit ou le grand modèle.
- **Mise en cache** : les questions répétées normalisées sont servies directement depuis un cache sans appel au modèle.

Le classificateur ici est volontairement simple. En production, vous le valideriez avec le trafic et pourriez le remplacer par le Routeur de Modèle de Foundry.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-5-nano
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-5-mini

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. L'agent, l'approbation humaine, et l'observabilité

Nous assemblons maintenant l'agent à partir des outils ci-dessus et enveloppons chaque requête dans un span OpenTelemetry. La fonction `handle_support_request` est le gestionnaire de requêtes en production : cache → route → trace → run → cache.

L'approbation humaine est gérée par le framework : parce que `issue_refund` est en `approval_mode="always_require"`, l'exécution se met en pause et affiche une demande d'approbation que nous résolvons explicitement.


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

# Build one agent per model tier so we can route by cost. The current agent-framework
# selects the model on the client, so each tier gets its own FoundryChatClient.
_TOOLS = [get_order_status, open_ticket, search_policies, issue_refund]
_agents_by_model: dict[str, object] = {}


def agent_for(model_name: str):
    if model_name not in _agents_by_model:
        client = FoundryChatClient(
            project_endpoint=endpoint,
            model=model_name,
            credential=AzureCliCredential(),
        )
        _agents_by_model[model_name] = client.as_agent(
            name="ContosoSupportAgent",
            instructions=SUPPORT_INSTRUCTIONS,
            tools=_TOOLS,
        )
    return _agents_by_model[model_name]


# Default agent (used by the evaluation gate, which does not route).
support_agent = agent_for(SMALL_MODEL)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await agent_for(chosen_model).run(prompt)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. Porte d'évaluation

Il s'agit de la porte de sortie de la leçon : un ensemble de tests hors ligne évalue l'agent, et le déploiement ne se fait que si le taux de réussite dépasse un seuil. Le scoreur ici est une simple vérification de recoupement de mots-clés pour que le notebook soit autonome ; en production, vous utiliseriez un LLM en tant que juge ou un évaluateur de cadre (voir la Leçon 10).


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## Mise en pratique : Une publication simulée

La cellule ci-dessous montre la boucle entière décrite dans la leçon : exécuter la porte d'évaluation, et ne « déployer » que si elle est validée. C’est le modèle que vous exécuteriez en CI avant de promouvoir une version d’agent vers le Foundry Agent Service.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## Résumé

Vous avez assemblé un agent de support client prêt pour la production avec chaque aspect opérationnel intégré :

- **Outils, RAG et mémoire** offrent à l’agent capacité et contexte.
- **Routage et mise en cache des modèles** maintiennent la latence et le coût sous contrôle.
- **Approbation humaine** protège les actions à haut risque comme les remboursements importants.
- **La porte d’évaluation** bloque les mauvaises versions avant leur déploiement.
- **Le traçage** rend chaque requête observable.

### Défi

Étendez cet agent pour :

1. **Supporter plusieurs modèles** — ajoutez un troisième niveau « raisonnement » et dirigez vers lui les escalades/plaintes.
2. **Ajouter des portes d’évaluation** — élargissez `TEST_CASES` pour inclure des scénarios d’approbation de remboursement et confirmez que la porte détecte les régressions.
3. **Ajouter un routage sensible au coût** — suivez un coût estimé par requête (petite vs grande vs cache) et affichez un rapport de coûts après un lot de requêtes mixtes.

Dans la leçon suivante, vous empruntez le chemin inverse et faites fonctionner un agent entièrement sur votre propre machine avec Microsoft Foundry Local et Qwen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :
Ce document a été traduit à l'aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçions d'assurer l'exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue native doit être considéré comme la source faisant autorité. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle réalisée par un humain. Nous ne saurions être tenus responsables des malentendus ou erreurs d'interprétation découlant de l'utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
